# Week 8 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-08.md`](../../weeks/week-08.md) · **Given engines:** [`supplychain.py`](supplychain.py) · [`data.py`](data.py)

Teaching walkthrough of the week-8 studio — *trust the artifact, not the name*. It
**imports the three attacks and three pins from [`solution.py`](solution.py)** —
never re-pasting them — so what runs here is exactly what `test_supplychain.py`
grades. Correctness is verified separately:
`python3 studios/_verify_solutions.py week-08`.

Pure-Python week: stdlib only, no `seclab`, no lab target, no Docker. Every value
is deterministic and copied from `data.py`, so every number below reproduces.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: week folder (for solution/supplychain/data) + repo root ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-08"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import data
import supplychain
import solution
print("loaded solution + engine; registries:", data.internal_registry, data.public_registry)

## Task 1 — Dependency confusion: the name-based resolver picks the attacker

`resolve_highest_version` considers **both** registries and picks the highest
version, **ignoring the source**. That omission *is* the bug: the attacker's
public `9.9.9` outranks your internal `1.4.2`, so their code installs under your
trusted name. Nobody typo'd anything — the resolver trusted a *name*.

In [ ]:
print(inspect.getsource(solution.resolve_highest_version))
src, ver = solution.resolve_highest_version("acme-auth")
print("naive resolve('acme-auth') ->", (src, ver))
# mirrors test_dependency_confusion_public_wins_until_pinned (first half)
assert (src, ver) == ("public", "9.9.9"), "attacker's public 9.9.9 must win the naive resolve"
print("attacker won:", src == "public")

### The pin — resolve against the lockfile (the fix)

`resolve_pinned` ignores the name-lookup entirely and checks the candidate
against the **lockfile**: the exact `source` **and** `sha256` you vetted. The
attacker's public artifact is refused; only the exact vetted internal artifact
installs. The name has become irrelevant.

In [ ]:
print(inspect.getsource(solution.resolve_pinned))
ok_attacker, why_a = solution.resolve_pinned("acme-auth", "evil99ff", "public")
ok_vetted,   why_v = solution.resolve_pinned("acme-auth", "aa11bb22cc", "internal")
print("attacker's public:", (ok_attacker, why_a))
print("vetted internal  :", (ok_vetted, why_v))
# mirrors test_dependency_confusion_public_wins_until_pinned (pin half)
assert not ok_attacker, "pinned resolver must REFUSE the attacker's public artifact"
assert ok_vetted, "pinned resolver must ACCEPT the exact vetted internal artifact"
print("\npin holds: only the exact vetted artifact installs")

## Task 2 — SBOM: you cannot pin, patch, or assess what you cannot enumerate

`build_sbom` flattens the **whole transitive tree**; `cve_affected` answers "are
we exposed?" as an instant set-intersection. `xz-utils` sits two levels deep —
nobody chose it, `acme-auth` pulled it in — which is exactly where the 2024
backdoor lived. Without an SBOM this question is days of grepping build logs.

In [ ]:
print(inspect.getsource(solution.build_sbom))
print(inspect.getsource(solution.cve_affected))
sbom = solution.build_sbom("our-app", data.dep_graph)
affected = solution.cve_affected(sbom, data.KNOWN_BAD)
print("SBOM    :", sbom)
print("affected:", affected)
# mirrors test_sbom_finds_transitive_cve
assert sbom == data.SBOM_EXPECTED, "SBOM mismatch"
assert "xz-utils" in sbom, "the transitive xz-utils dep must appear"
assert affected == data.AFFECTED_EXPECTED, "CVE query mismatch"
print("\ntransitive xz-utils surfaced; CVE query answered instantly")

## Task 3 — Watch the `alg=none` guarantee COLLAPSE (the attack moment)

`verify_insecure` (given) promises *"only a token signed under our key is valid."*
But it reads the token's **own `alg` header** and obeys it. `forge_alg_none_admin`
sets `alg=none`, `role=admin`, and an **empty signature** — no signing key, no
crypto broken. Watch it hand over admin live.

In [ ]:
print(inspect.getsource(solution.forge_alg_none_admin))
from supplychain import verify_insecure, make_token
forged = solution.forge_alg_none_admin()
claims = verify_insecure(forged)                 # the naive verifier ACCEPTS it
print("forged token:", forged)
print("verify_insecure accepts it as:", claims)
# mirrors test_alg_none_forgery_accepted_until_algorithm_pinned
assert claims is not None and claims.get("role") == "admin", "alg=none must escalate to admin"
print("\n*** GUARANTEE COLLAPSED: unsigned token accepted as admin, no key needed ***")

### The pin — verify with a **server-side** algorithm (the fix)

`verify_secure` never trusts the token's header: it rejects any `alg` that is not
the one the *server* pinned (so `alg=none` dies immediately), then checks the
HS256 signature under `SECRET`. The forgery is rejected; a legitimately signed
token still verifies.

In [ ]:
print(inspect.getsource(solution.verify_secure))
legit = make_token(data.LEGIT_PAYLOAD)
print("pinned verifier, forgery:", solution.verify_secure(forged))
print("pinned verifier, legit  :", solution.verify_secure(legit))
# mirrors test_algorithm_pinning_rejects_forgery
assert solution.verify_secure(forged) is None, "algorithm pin must REJECT the alg=none forgery"
assert solution.verify_secure(legit) == data.LEGIT_PAYLOAD, "pin must still ACCEPT a real token"
print("\nalgorithm pinning rejected the forgery, accepted the real token")

## The lesson — Control Scorecard axis 2 (trust the artifact, not the name)

All three attacks are one shape, and so is the fix. The scorecard distinction is
**GUARANTEE vs RAISES-COST**:

| Attack | Trusted *name* | Substituted *artifact* | The pin | Bucket |
|---|---|---|---|---|
| Dependency confusion | package name + version | attacker's public package | lockfile + `sha256` + source scope | **GUARANTEE** — only the exact vetted artifact installs |
| Transitive backdoor (XZ) | "it's just a dependency" | compromised upstream code | SBOM → pin/patch enumerated deps | **GUARANTEE (conditional)** — *if* it's in the SBOM you can act on it |
| `alg=none` JWT | the token's own `alg` header | an unsigned token | server-side algorithm pinning | **GUARANTEE** — verified under a value *you* chose |
| "keep the algorithm secret" | — | — | security-through-obscurity | **RAISES COST only** — one leak and it's gone |

The pin is a guarantee because the trusted set is a value **you** control (a hash,
an algorithm, a lockfile) — a promise a name-based lookup can never make. Same
pattern returns in week 10's tool allow-list, with weaker guarantees each time.

In [ ]:
print("LESSON: a name-based trust decision is never a guarantee — watch it break once (alg=none",
      "handed us admin) to prove it. Pinning the ARTIFACT (hash / algorithm / lockfile) under a",
      "value YOU control is the guarantee. Trust the artifact, not the name.")